# 01 — PLSA baseline on the 20 Newsgroups corpus

This experiment examines whether Probabilistic Latent Semantic Analysis (PLSA) can learn useful topics from the 20 Newsgroups collection and how PLSA retrieval compares with TF-IDF and Latent Semantic Indexing (LSI). It also probes how PLSA represents documents made from mixtures of two categories.

The shared corpus is downloaded and preprocessed in `src/preprocessing.ipynb`. Here, we define this experiment's train/test split, fit representations on training data, evaluate retrieval on held-out documents, and save the results.

Run this notebook from the `Local` repository root. Its outputs are written to `results/plsa_baseline/`; each run replaces that experiment's previous output files.

## 1. Imports, paths and experiment settings

We use a fixed random seed, a stratified 75/25 train/test split, five PLSA topics, five random initializations, and a maximum of 100 EM iterations. The count and TF-IDF vocabularies are capped at 3,000 terms.

In [ ]:
import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from IPython.display import display

REPOSITORY_ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.evaluation import evaluate_ranked_retrieval
from src.lsi import fit_lsi
from src.plsa_model import PLSA, train_plsa_restarts
from src.retrieval import cosine_scores
from src.synthetic_data import make_mixed_document
from src.text_processing import load_nlp, preprocess_texts
from src.tfidf import fit_tfidf

SEED = 42
K = 5
N_RESTARTS = 5
MAX_ITER = 100
TOLERANCE = 1e-4
TEST_SIZE = 0.25
COUNT_MAX_FEATURES = 3000
COUNT_MIN_DF = 3
COUNT_MAX_DF = 0.90
TOP_K = 10

DATA_PATH = REPOSITORY_ROOT / 'data' / '20news_preprocessed.npz'
RESULTS_ROOT = REPOSITORY_ROOT / 'results' / 'plsa_baseline'
OUTPUT_DIR = RESULTS_ROOT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repository: {REPOSITORY_ROOT.resolve()}')
print(f'Results will be replaced at: {OUTPUT_DIR.resolve()}')

## 2. Load the corpus and define the train/test split

The prepared corpus contains cleaned document text, original category labels, and category names. We create a stratified split before fitting count, TF-IDF, or LSI representations. Those representations are fitted on training text only and then applied to the held-out test set.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Missing {DATA_PATH}. Run src/preprocessing.ipynb first.'
    )
with np.load(DATA_PATH, allow_pickle=False) as corpus:
    documents = corpus['documents'].tolist()
    labels = corpus['labels'].astype(np.int64, copy=False)
    target_names = corpus['category_names'].tolist()

if len(documents) != len(labels):
    raise ValueError('The processed corpus and labels have different lengths.')

print(f'Documents: {len(documents):,}')
print(f'Categories: {len(target_names)}')
print(f'Empty processed documents: {sum(not text.strip() for text in documents):,}')
display(pd.DataFrame({
    'category': target_names,
    'documents': np.bincount(labels, minlength=len(target_names)),
}))

docs_train, docs_test, y_train, y_test = train_test_split(
    documents, labels, test_size=TEST_SIZE, random_state=SEED, stratify=labels
)
print(f'Train documents: {len(docs_train):,}; test documents: {len(docs_test):,}')

## 3. Fit the count vocabulary on training documents

The count representation uses whitespace tokenization, at most 3,000 features, `min_df=3`, and `max_df=0.90`. We fit the vocabulary on training documents and use it to transform test documents. Empty documents remain in the split.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(
    lowercase=False, tokenizer=str.split, token_pattern=None,
    max_features=COUNT_MAX_FEATURES, min_df=COUNT_MIN_DF,
    max_df=COUNT_MAX_DF, dtype=np.float64,
)
X_train = count_vectorizer.fit_transform(docs_train)
X_test = count_vectorizer.transform(docs_test)
vocabulary = count_vectorizer.get_feature_names_out()

print('X_train:', X_train.shape, '(documents, vocabulary)')
print('X_test: ', X_test.shape)
print('Example vocabulary:', vocabulary[:20])
print('Empty train documents:', sum(not text.strip() for text in docs_train))
print('Empty test documents: ', sum(not text.strip() for text in docs_test))

## 4. Fit PLSA with multiple random initializations

For document `d`, word `w`, and latent topic `z`, PLSA models `P(w | d) = sum_z P(w | z) P(z | d)`. We fit a five-topic model with the EM algorithm from several random initializations and select the run with the highest training log-likelihood.

In [ ]:
best_plsa, restart_runs = train_plsa_restarts(
    X_train, n_topics=K, n_restarts=N_RESTARTS,
    max_iter=MAX_ITER, tol=TOLERANCE, base_seed=SEED,
)
theta_train = best_plsa.transform()
theta_test = best_plsa.fold_in(X_test, max_iter=100)
print('theta_train:', theta_train.shape)
print('theta_test: ', theta_test.shape)

restart_summary = pd.DataFrame([
    {
        'restart': i + 1,
        'seed': SEED + i,
        'iterations': model.n_iter_,
        'training_log_likelihood': float(final_ll),
        'selected': model is best_plsa,
    }
    for i, (model, final_ll) in enumerate(restart_runs)
])
display(restart_summary)
restart_summary.to_csv(OUTPUT_DIR / 'restart_summary.csv', index=False)

## 5. Fit TF-IDF and LSI baselines

TF-IDF learns its vocabulary and inverse-document-frequency weights from training documents. LSI applies truncated SVD to the training TF-IDF matrix and normalizes projected rows. We transform test documents and queries with the fitted training objects.

In [ ]:
tfidf_vectorizer, (X_train_tfidf, X_test_tfidf) = fit_tfidf(
    docs_train, docs_test,
    max_features=COUNT_MAX_FEATURES, min_df=COUNT_MIN_DF, max_df=COUNT_MAX_DF,
)
lsi_model, (lsi_train, lsi_test) = fit_lsi(
    X_train_tfidf, X_test_tfidf, n_components=50, seed=SEED,
)
print('TF-IDF train/test:', X_train_tfidf.shape, X_test_tfidf.shape)
print('LSI train/test:   ', lsi_train.shape, lsi_test.shape)

## 6. Inspect learned topics and convergence

We inspect the most probable words in each learned topic and plot training log-likelihood over EM iterations. Topics are latent components and are not assumed to correspond one-to-one with newsgroup categories.

In [ ]:
topic_rows = []
for topic_id, words in best_plsa.top_words(vocabulary, n=12).items():
    print(f'\nTopic {topic_id}:')
    print(', '.join(f'{word} ({prob:.3f})' for word, prob in words))
    for rank, (word, probability) in enumerate(words, start=1):
        topic_rows.append({
            'topic': topic_id, 'rank': rank, 'word': word,
            'probability': probability,
        })
topic_words = pd.DataFrame(topic_rows)
topic_words.to_csv(OUTPUT_DIR / 'topic_words.csv', index=False)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(best_plsa.log_likelihoods) + 1),
         best_plsa.log_likelihoods, marker='o', markersize=3)
plt.xlabel('EM iteration')
plt.ylabel('Training log-likelihood')
plt.title('PLSA convergence')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plsa_convergence.png', dpi=160, bbox_inches='tight')
plt.show()
pd.DataFrame({
    'iteration': np.arange(1, len(best_plsa.log_likelihoods) + 1),
    'training_log_likelihood': best_plsa.log_likelihoods,
}).to_csv(OUTPUT_DIR / 'plsa_convergence.csv', index=False)

## 7. Compare retrieval on held-out test documents

We use one fixed keyword query for each category. A test document is relevant when its category matches the query's target category. We report Precision@10, Recall@10, AP@10, and NDCG@10 for each query and the mean across the query set.

In [ ]:
category_queries = {
    'alt.atheism': 'atheism atheist religion belief',
    'comp.graphics': 'computer graphics rendering image pixels',
    'comp.os.ms-windows.misc': 'windows operating system microsoft files',
    'comp.sys.ibm.pc.hardware': 'IBM PC hardware disk memory computer',
    'comp.sys.mac.hardware': 'Apple Macintosh Mac hardware computer',
    'comp.windows.x': 'X Window System display server windows',
    'misc.forsale': 'for sale price shipping offer',
    'rec.autos': 'car automobile engine dealer',
    'rec.motorcycles': 'motorcycle bike riding helmet',
    'rec.sport.baseball': 'baseball players team season game',
    'rec.sport.hockey': 'hockey team NHL puck season',
    'sci.crypt': 'cryptography encryption key security',
    'sci.electronics': 'electronics circuit voltage resistor',
    'sci.med': 'medical doctor disease treatment health',
    'sci.space': 'space NASA spacecraft orbit planet',
    'soc.religion.christian': 'Christian church Bible Jesus faith',
    'talk.politics.guns': 'gun control firearms second amendment',
    'talk.politics.mideast': 'Middle East Israel Arab Palestinian',
    'talk.politics.misc': 'government election political policy',
    'talk.religion.misc': 'religion religious belief faith',
}
missing_queries = sorted(set(target_names) - set(category_queries))
extra_queries = sorted(set(category_queries) - set(target_names))
if missing_queries or extra_queries:
    raise ValueError(f'Query/category mismatch. Missing={missing_queries}; extra={extra_queries}')

query_nlp = load_nlp()
def evaluate_query(query_text, target_label, target_category, k=TOP_K):
    processed_query = preprocess_texts([query_text], nlp=query_nlp)[0]
    if not processed_query.strip():
        print(f'Warning: query for {target_category} became empty after preprocessing.')
    q_counts = count_vectorizer.transform([processed_query]).astype(np.float64)
    q_tfidf = tfidf_vectorizer.transform([processed_query])
    q_lsi = lsi_model.transform(q_tfidf)
    q_plsa = best_plsa.fold_in(q_counts, max_iter=100)
    method_scores = {
        'TF-IDF/VSM': cosine_scores(q_tfidf, X_test_tfidf),
        'LSI': cosine_scores(q_lsi, lsi_test),
        'PLSA': cosine_scores(q_plsa, theta_test),
    }
    rows = []
    for method, scores in method_scores.items():
        metrics = evaluate_ranked_retrieval(scores, y_test, target_label, k=k)
        rows.append({
            'query': query_text, 'target_category': target_category,
            'method': method, **metrics,
            'relevant_test_documents': int(np.count_nonzero(y_test == target_label)),
        })
    return rows

evaluation_rows = []
for category in target_names:
    evaluation_rows.extend(evaluate_query(
        category_queries[category], target_names.index(category), category, k=TOP_K
    ))
evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)
evaluation_df.to_csv(OUTPUT_DIR / 'retrieval_metrics_by_query.csv', index=False)

metric_columns = ['Precision@k', 'Recall@k', 'NDCG@k', 'AP@k']
mean_metrics = evaluation_df.groupby('method')[metric_columns].mean()
mean_metrics = mean_metrics.rename(columns={'AP@k': 'MAP@k'})
display(mean_metrics)
mean_metrics.to_csv(OUTPUT_DIR / 'retrieval_metrics_mean.csv')

In [ ]:
mean_metrics.plot(kind='bar', figsize=(9, 5), ylim=(0, 1), rot=0)
plt.title('Held-out retrieval comparison')
plt.ylabel('Score')
plt.xlabel('Method')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'retrieval_metrics_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

## 8. Follow up: examine PLSA topic mixtures

The retrieval comparison answers how the methods rank held-out documents. We now ask a different question about PLSA itself: when a document contains words sampled from two categories, do its inferred topic probabilities respond to the mixture?

We create artificial documents from `rec.sport.baseball` and `talk.politics.misc` training documents at several token proportions, then fold each mixture into the fitted PLSA model. We identify each category's most associated topic from its average training-document topic mixture. A learned topic may combine categories, so the inferred topic probabilities need not equal the source-token proportions.

In [ ]:
sports_category = 'rec.sport.baseball'
politics_category = 'talk.politics.misc'
sports_class = target_names.index(sports_category)
politics_class = target_names.index(politics_category)

class_topic_means = np.vstack([
    theta_train[y_train == category_id].mean(axis=0)
    for category_id in range(len(target_names))
])
sports_topic = int(class_topic_means[sports_class].argmax())
politics_topic = int(class_topic_means[politics_class].argmax())
print('Sports-associated topic:', sports_topic)
print('Politics-associated topic:', politics_topic)
if sports_topic == politics_topic:
    print('Warning: both categories map to the same dominant topic; interpret cautiously.')

mixture_rng = np.random.default_rng(SEED + 77)
ratios = [0.9, 0.7, 0.5, 0.3, 0.1]
mixture_rows = []
for ratio in ratios:
    for replicate in range(20):
        mixed_counts = make_mixed_document(
            X_train, y_train, sports_class, politics_class,
            ratio, mixture_rng, n_tokens=300,
        )
        estimate = best_plsa.fold_in(mixed_counts, max_iter=100)[0]
        mixture_rows.append({
            'source_category_a': sports_category,
            'source_category_b': politics_category,
            'actual_sports_word_ratio': ratio,
            'actual_politics_word_ratio': 1 - ratio,
            'estimated_sports_topic': estimate[sports_topic],
            'estimated_politics_topic': estimate[politics_topic],
            'replicate': replicate,
        })

mixture_df = pd.DataFrame(mixture_rows)
mixture_summary = mixture_df.groupby('actual_sports_word_ratio').agg(
    estimated_sports_mean=('estimated_sports_topic', 'mean'),
    estimated_sports_std=('estimated_sports_topic', 'std'),
    estimated_politics_mean=('estimated_politics_topic', 'mean'),
    estimated_politics_std=('estimated_politics_topic', 'std'),
).reset_index()
display(mixture_summary)
mixture_df.to_csv(OUTPUT_DIR / 'synthetic_mixture_replicates.csv', index=False)
mixture_summary.to_csv(OUTPUT_DIR / 'synthetic_mixture_summary.csv', index=False)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot([0, 1], [0, 1], 'k--', label='Ideal reference')
plt.errorbar(
    mixture_summary['actual_sports_word_ratio'],
    mixture_summary['estimated_sports_mean'],
    yerr=mixture_summary['estimated_sports_std'],
    marker='o', capsize=3, label='Estimated sports-topic probability',
)
plt.xlabel('Actual sports word-token proportion')
plt.ylabel('Estimated sports-associated topic probability')
plt.title('PLSA topic mixtures for synthetic documents')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'synthetic_mixture_topic_probabilities.png', dpi=160, bbox_inches='tight')
plt.show()

## 9. Save outputs

The experiment tables, figures, and configuration are saved in `results/plsa_baseline/`. 

In [ ]:
run_metadata = {
    'experiment': 'PLSA baseline on all 20 Newsgroups categories',
    'seed': SEED, 'test_size': TEST_SIZE, 'n_topics': K,
    'n_restarts': N_RESTARTS, 'max_iter': MAX_ITER,
    'tolerance': TOLERANCE, 'count_max_features': COUNT_MAX_FEATURES,
    'count_min_df': COUNT_MIN_DF, 'count_max_df': COUNT_MAX_DF,
    'retrieval_k': TOP_K, 'dataset_documents': len(documents),
    'dataset_categories': target_names,
    'synthetic_categories': [sports_category, politics_category],
    }
with (OUTPUT_DIR / 'configuration.json').open('w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2)
print(f'Baseline outputs and configuration saved under {OUTPUT_DIR.resolve()}')